# State Management

এতক্ষণ আমরা state-কে বেশ সাধারণভাবে ব্যবহার করেছি। কিন্তু বাস্তব AI system-এ state অনেক জটিল হয় — list-এ item যোগ হয়, একাধিক node একসাথে state update করতে পারে, ইত্যাদি। LangGraph-এ এই জটিল scenario সামলানোর জন্য আছে **Reducers** এবং **Annotated types**।

## এই notebook-এ যা শিখব:

| বিষয় | বিবরণ |
|---|---|
| State overwrite সমস্যা | কেন সাধারণ list বিপদজনক |
| Reducer | কীভাবে state update হবে সেটা define করা |
| `Annotated` | Python type hint দিয়ে reducer যোগ করা |
| `operator.add` | List merge করার built-in reducer |
| Custom Reducer | নিজের নিয়মে state update |

**মূল সমস্যা যেটা solve করব:** দুটো node যদি একই field update করে, কোনটা জেতে? Reducer সেটা নির্ধারণ করে।

In [7]:
from typing import TypedDict, Annotated, List
import operator
from langgraph.graph import StateGraph, END

## সমস্যাটা কী? — Overwrite বিপদ

প্রথমে দেখি সাধারণ list দিয়ে কী সমস্যা হয়। ধরো দুটো node একই `messages` list-এ item যোগ করতে চায়:

```python
# Node A করে:
state['messages'] = ["Node A-র message"]

# Node B করে:
state['messages'] = ["Node B-র message"]

# ফলাফল: শুধু ["Node B-র message"] থাকে! Node A-র টা মুছে যায়! 😱
```

এটাই overwrite সমস্যা। Reducer দিয়ে বলা যায় — "নতুন value দিয়ে **replace করো না**, বরং **যোগ করো**"।

## Reducer কী?

Reducer হলো একটা function যেটা বলে দেয় state কীভাবে update হবে:

```python
def my_reducer(existing_value, new_value):
    # existing = এখন যা আছে
    # new = node যা return করেছে
    return existing_value + new_value  # দুটো মিলিয়ে দাও
```

LangGraph-এ `operator.add` হলো সবচেয়ে common reducer — list বা string যোগ করার জন্য।

```python
# Annotated দিয়ে field-এ reducer assign করা:
messages: Annotated[List[str], operator.add]
#          ↑ type          ↑ reducer
```

In [8]:
# সাধারণ state — overwrite হয়
class SimpleState(TypedDict):
    messages: List[str]  # ⚠️ overwrite হবে!


# Reducer সহ state — যোগ হয়
class BetterState(TypedDict):
    messages: Annotated[List[str], operator.add]  # ✅ append হবে!
    count: int  # এটা সাধারণ — overwrite হবে (কিন্তু এক্ষেত্রে ঠিকই আছে)

## পার্থক্য দেখা — Simple vs Reducer State

In [9]:
# ── সাধারণ State (সমস্যা আছে) ───────────────────────────────
print("❌ সাধারণ State (overwrite সমস্যা):")

def node_a_simple(state: SimpleState) -> SimpleState:
    return {"messages": ["Node A থেকে message"]}  # শুধু A-র message

def node_b_simple(state: SimpleState) -> SimpleState:
    return {"messages": ["Node B থেকে message"]}  # শুধু B-র message — A-টা মুছে যাবে!

g_simple = StateGraph(SimpleState)
g_simple.add_node("a", node_a_simple)
g_simple.add_node("b", node_b_simple)
g_simple.set_entry_point("a")
g_simple.add_edge("a", "b")
g_simple.add_edge("b", END)

app_simple = g_simple.compile()
result_simple = app_simple.invoke({"messages": []})
print(f"ফলাফল: {result_simple['messages']}")
print("Node A-র message নেই! 😱\n")

❌ সাধারণ State (overwrite সমস্যা):
ফলাফল: ['Node B থেকে message']
Node A-র message নেই! 😱



In [10]:
# ── Reducer সহ State (সমাধান) ─────────────────────────────────
print("✅ Reducer সহ State (সঠিক):")

def node_a_better(state: BetterState) -> dict:
    # শুধু নতুন message return করি — LangGraph নিজেই যোগ করে নেবে
    return {"messages": ["Node A থেকে message"], "count": 1}

def node_b_better(state: BetterState) -> dict:
    return {"messages": ["Node B থেকে message"], "count": state['count'] + 1}

g_better = StateGraph(BetterState)
g_better.add_node("a", node_a_better)
g_better.add_node("b", node_b_better)
g_better.set_entry_point("a")
g_better.add_edge("a", "b")
g_better.add_edge("b", END)

app_better = g_better.compile()
result_better = app_better.invoke({"messages": [], "count": 0})
print(f"ফলাফল: {result_better['messages']}")
print(f"Count: {result_better['count']}")
print("দুটো message-ই আছে! 🎉")

✅ Reducer সহ State (সঠিক):
ফলাফল: ['Node A থেকে message', 'Node B থেকে message']
Count: 2
দুটো message-ই আছে! 🎉


## Custom Reducer তৈরি করা

কখনো কখনো `operator.add` যথেষ্ট না। তখন নিজের reducer লিখতে হয়। যেমন:
- শুধু unique item রাখতে চাই
- নির্দিষ্ট সংখ্যক item-এর পর পুরনোটা বাদ দিতে চাই
- বিশেষ নিয়মে merge করতে চাই

In [11]:
# Custom Reducer 1: Unique items only
def unique_merge(existing: List[str], new: List[str]) -> List[str]:
    """duplicate বাদ দিয়ে merge করে"""
    combined = existing + new
    # Order বজায় রেখে unique করা
    seen = set()
    result = []
    for item in combined:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result


# Custom Reducer 2: শুধু শেষ N টা রাখো
def keep_last_3(existing: List[str], new: List[str]) -> List[str]:
    """সর্বোচ্চ ৩টা item রাখে"""
    combined = existing + new
    return combined[-3:]  # শেষ ৩টা


class UniqueState(TypedDict):
    tags: Annotated[List[str], unique_merge]  # duplicate থাকবে না
    recent: Annotated[List[str], keep_last_3]  # সর্বোচ্চ ৩টা


def add_tags_1(state: UniqueState) -> dict:
    return {
        "tags": ["python", "ai", "langgraph"],
        "recent": ["event_1"]
    }

def add_tags_2(state: UniqueState) -> dict:
    return {
        "tags": ["ai", "langchain", "python"],  # 'ai', 'python' duplicate!
        "recent": ["event_2", "event_3", "event_4"]
    }

def add_tags_3(state: UniqueState) -> dict:
    return {
        "tags": ["new_tag"],
        "recent": ["event_5"]
    }

g3 = StateGraph(UniqueState)
g3.add_node("step1", add_tags_1)
g3.add_node("step2", add_tags_2)
g3.add_node("step3", add_tags_3)
g3.set_entry_point("step1")
g3.add_edge("step1", "step2")
g3.add_edge("step2", "step3")
g3.add_edge("step3", END)

app3 = g3.compile()
result3 = app3.invoke({"tags": [], "recent": []})

print("Custom Reducer ফলাফল:")
print(f"Tags (unique): {result3['tags']}")
print(f"Recent (শেষ ৩টা): {result3['recent']}")

Custom Reducer ফলাফল:
Tags (unique): ['python', 'ai', 'langgraph', 'langchain', 'new_tag']
Recent (শেষ ৩টা): ['event_3', 'event_4', 'event_5']


## বাস্তব উদাহরণ: Chat History State

AI chatbot-এ সবচেয়ে বেশি যেটা দরকার — conversation history রাখা। এখন সেটা সঠিকভাবে করা যাবে।

In [12]:
from typing import Dict

# Message dictionary-এর list — chat history এভাবেই রাখা হয়
def merge_messages(existing: List[Dict], new: List[Dict]) -> List[Dict]:
    """নতুন messages পুরনো history-তে যোগ করে"""
    return existing + new


class ChatState(TypedDict):
    messages: Annotated[List[Dict], merge_messages]  # conversation history
    user_name: str
    turn_count: int


def greet_node(state: ChatState) -> dict:
    greeting = f"আস-সালামু-আলাইকুম, {state['user_name']}! আমি আপনাকে সাহায্য করতে প্রস্তুত।"
    return {
        "messages": [{"role": "assistant", "content": greeting}],
        "turn_count": state['turn_count'] + 1
    }

def info_node(state: ChatState) -> dict:
    info = f"আপনার সাথে কথা বলতে পেরে ভালো লাগছে! এটি আমাদের {state['turn_count'] + 1} নম্বর বার্তা।"
    return {
        "messages": [{"role": "assistant", "content": info}],
        "turn_count": state['turn_count'] + 1
    }

def farewell_node(state: ChatState) -> dict:
    farewell = f"ধন্যবাদ {state['user_name']}! মোট {state['turn_count'] + 1}টি বার্তা হয়েছে। আল্লাহ হাফেজ!"
    return {
        "messages": [{"role": "assistant", "content": farewell}],
        "turn_count": state['turn_count'] + 1
    }


chat_graph = StateGraph(ChatState)
chat_graph.add_node("greet", greet_node)
chat_graph.add_node("info", info_node)
chat_graph.add_node("farewell", farewell_node)
chat_graph.set_entry_point("greet")
chat_graph.add_edge("greet", "info")
chat_graph.add_edge("info", "farewell")
chat_graph.add_edge("farewell", END)

chat_app = chat_graph.compile()

# Initial state — user-এর প্রথম message সহ
initial = {
    "messages": [{"role": "user", "content": "হ্যালো!"}],
    "user_name": "রাসেল",
    "turn_count": 0
}

result = chat_app.invoke(initial)

print("💬 পুরো Conversation History:")
print("-" * 50)
for msg in result['messages']:
    role = "👤 User" if msg['role'] == 'user' else "🤖 Assistant"
    print(f"{role}: {msg['content']}")
print("-" * 50)
print(f"মোট বার্তা: {len(result['messages'])}")

💬 পুরো Conversation History:
--------------------------------------------------
👤 User: হ্যালো!
🤖 Assistant: আস-সালামু-আলাইকুম, রাসেল! আমি আপনাকে সাহায্য করতে প্রস্তুত।
🤖 Assistant: আপনার সাথে কথা বলতে পেরে ভালো লাগছে! এটি আমাদের 2 নম্বর বার্তা।
🤖 Assistant: ধন্যবাদ রাসেল! মোট 3টি বার্তা হয়েছে। আল্লাহ হাফেজ!
--------------------------------------------------
মোট বার্তা: 4


## সারসংক্ষেপ

এই notebook-এ শিখলাম:

- ✅ **Overwrite সমস্যা** — সাধারণ list field একটা node শেষে আরেকটা node overwrite করে
- ✅ **Reducer** — state কীভাবে update হবে সেটার নিয়ম
- ✅ **Annotated[type, reducer]** — field-এ reducer যোগ করার syntax
- ✅ **operator.add** — list append করার সহজ built-in reducer
- ✅ **Custom Reducer** — নিজের merge logic লেখা

**মূল সূত্র:**
```python
# সাধারণ field — overwrite হয়
name: str

# Reducer সহ — accumulate হয়
messages: Annotated[List[str], operator.add]
```

**পরের notebook:** LLM Integration — এবার আসল AI! ChatGPT/Claude-এর সাথে LangGraph জোড়া লাগাব! 🤖